# Avaliação do LLaVA no IAM Handwriting

**Fluxo:**
1. Instalar dependências
2. Baixar e montar o dataset IAM
3. Configurar e carregar o modelo
4. Rodar **inferência + métricas + análise de erros + exportação** em uma única célula

> **Modelo padrão:** `YouLiXiya/tinyllava-v1.0-1.1b-hf`  
> Ele é menor e mais viável no Colab do que `llava-hf/llava-1.5-7b-hf`.  
> Se quiser comparar com o LLaVA 7B depois, basta trocar `MODEL_ID`.


## 1. Instalar dependências

In [1]:
!pip install -q kagglehub transformers accelerate evaluate jiwer tqdm rapidfuzz bitsandbytes

## 2. Imports e utilitários

In [2]:

from pathlib import Path
import os
import glob
import json
import re
import time
import warnings

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import evaluate
from rapidfuzz.distance import Levenshtein
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)


## 3. Baixar o dataset IAM via KaggleHub

In [3]:

import kagglehub

dataset_root = kagglehub.dataset_download("changheonkim/iam-trocr")
dataset_root = Path(dataset_root) / "IAM"

image_directory = dataset_root / "image"
image_paths = sorted(glob.glob(str(image_directory / "*.jpg")))

print("Dataset root:", dataset_root)
print("Total de imagens encontradas:", len(image_paths))
print("Primeiras 3 imagens:", image_paths[:3])


Using Colab cache for faster access to the 'iam-trocr' dataset.
Dataset root: /kaggle/input/iam-trocr/IAM
Total de imagens encontradas: 2915
Primeiras 3 imagens: ['/kaggle/input/iam-trocr/IAM/image/c04-110-00.jpg', '/kaggle/input/iam-trocr/IAM/image/c04-110-01.jpg', '/kaggle/input/iam-trocr/IAM/image/c04-110-02.jpg']


## 4. Encontrar e carregar labels

In [4]:

def find_annotation_files(base: Path):
    exts = (".csv", ".tsv", ".jsonl", ".json", ".txt")
    files = []
    for root, _, fnames in os.walk(base):
        for f in fnames:
            if f.lower().endswith(exts):
                files.append(Path(root) / f)
    return files

def build_labels_map(base: Path):
    ann_files = find_annotation_files(base)
    print("Arquivos candidatos a annotation:")
    for f in ann_files[:30]:
        print(" -", f.relative_to(base))
    if len(ann_files) > 30:
        print(f" ... +{len(ann_files)-30} outros")

    labels = {}

    # CSV / TSV
    for f in ann_files:
        if f.suffix.lower() in [".csv", ".tsv"]:
            sep = "\t" if f.suffix.lower() == ".tsv" else ","
            try:
                df_ = pd.read_csv(f, sep=sep)
            except Exception:
                continue

            cols = [c.lower() for c in df_.columns]
            img_candidates = ["image", "image_path", "filename", "file", "img", "path"]
            txt_candidates = ["text", "gt", "label", "transcript", "annotation", "sentence"]

            img_col = next((df_.columns[i] for i, c in enumerate(cols) if c in img_candidates), None)
            txt_col = next((df_.columns[i] for i, c in enumerate(cols) if c in txt_candidates), None)

            if img_col and txt_col:
                for _, r in df_.iterrows():
                    img = str(r[img_col])
                    txt = str(r[txt_col])
                    labels[Path(img).stem] = txt
                if labels:
                    print("Labels carregados de:", f.relative_to(base))
                    return labels

    # JSONL
    for f in ann_files:
        if f.suffix.lower() == ".jsonl":
            try:
                with open(f, "r", encoding="utf-8") as fh:
                    for line in fh:
                        obj = json.loads(line)
                        img = obj.get("image") or obj.get("image_path") or obj.get("filename") or obj.get("file")
                        txt = obj.get("text") or obj.get("gt") or obj.get("label") or obj.get("transcript")
                        if img and txt is not None:
                            labels[Path(img).stem] = str(txt)
                if labels:
                    print("Labels carregados de:", f.relative_to(base))
                    return labels
            except Exception:
                labels = {}

    # TXT
    for f in ann_files:
        if f.suffix.lower() == ".txt":
            try:
                with open(f, "r", encoding="utf-8") as fh:
                    for line in fh:
                        line = line.strip()
                        if not line:
                            continue
                        if "\t" in line:
                            a, b = line.split("\t", 1)
                        else:
                            parts = re.split(r"\s{2,}", line, maxsplit=1)
                            if len(parts) < 2:
                                continue
                            a, b = parts[0], parts[1]
                        labels[Path(a).stem] = b.strip()
                if labels:
                    print("Labels carregados de:", f.relative_to(base))
                    return labels
            except Exception:
                labels = {}

    raise RuntimeError("Não foi possível localizar um arquivo de labels compatível dentro de IAM/.")

labels_map = build_labels_map(dataset_root)
print("Quantidade de labels carregadas:", len(labels_map))


Arquivos candidatos a annotation:
 - gt_test.txt
 - gpt2.dict.txt
Labels carregados de: gt_test.txt
Quantidade de labels carregadas: 2915


## 5. Montar dataframe base

In [5]:

rows = []
missing = 0

for p in image_paths:
    stem = Path(p).stem
    gt = labels_map.get(stem)
    if gt is None:
        missing += 1
        continue
    rows.append({
        "id": stem,
        "image_path": p,
        "gt": gt,
    })

df = pd.DataFrame(rows)

print("Total de imagens:", len(image_paths))
print("Com label:", len(df))
print("Sem label:", missing)
display(df.head())


Total de imagens: 2915
Com label: 2915
Sem label: 0


,id,image_path,gt
0,c04-110-00,/kaggle/input/iam-trocr/IAM/image/c04-110-00.jpg,Become a success with a disc and hey presto ! You're a star ... . Rolly sings with
1,c04-110-01,/kaggle/input/iam-trocr/IAM/image/c04-110-01.jpg,"assuredness "" Bella Bella Marie "" ( Parlophone ) , a lively song that changes tempo mid-way ."
2,c04-110-02,/kaggle/input/iam-trocr/IAM/image/c04-110-02.jpg,"I don't think he will storm the charts with this one , but it's a good start ."
3,c04-110-03,/kaggle/input/iam-trocr/IAM/image/c04-110-03.jpg,"CHRIS CHARLES , 39 , who lives in Stockton-on-Tees , is an accountant ."
4,c04-116-00,/kaggle/input/iam-trocr/IAM/image/c04-116-00.jpg,He is also a director of a couple of garages . And he finds time as well to be a lyric


## 6. Configuração do experimento

Se estiver **sem GPU**, mantenha `SUBSET_N` pequeno para o notebook terminar.  
Se estiver **com GPU**, pode aumentar o subset.


In [6]:
EXPERIMENT_NAME = "LLaVA on IAM"
MODEL_ID = "llava-hf/llava-1.5-7b-hf"
PROMPT = "Transcribe exactly the handwritten text in this image. Return only the transcription, with no explanation."
MAX_NEW_TOKENS = 64
SUBSET_N = 30
RANDOM_STATE = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device detectado:", device)

if device == "cuda":
    SUBSET_N = 300
else:
    SUBSET_N = 30
    print("Sem GPU: usando subset menor para evitar execução muito longa.")


Device detectado: cuda


## 7. Carregar modelo e processor

In [7]:
if device == "cuda":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto",
    )
else:
    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    ).to(device)

processor = AutoProcessor.from_pretrained(MODEL_ID)

if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "left"

# Corrige campos que podem vir ausentes em alguns checkpoints TinyLLaVA
vision_config = getattr(model.config, "vision_config", None)

if getattr(processor, "patch_size", None) is None:
    if vision_config is not None and getattr(vision_config, "patch_size", None) is not None:
        processor.patch_size = vision_config.patch_size
    else:
        processor.patch_size = 14

if getattr(processor, "num_additional_image_tokens", None) is None:
    processor.num_additional_image_tokens = 1

if getattr(processor, "vision_feature_select_strategy", None) is None:
    processor.vision_feature_select_strategy = "default"

print("patch_size:", processor.patch_size)
print("num_additional_image_tokens:", processor.num_additional_image_tokens)
print("vision_feature_select_strategy:", processor.vision_feature_select_strategy)

df_run = df.sample(min(SUBSET_N, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
print("Sample size:", len(df_run))
display(df_run.head())

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


patch_size: 14
num_additional_image_tokens: 1
vision_feature_select_strategy: default
Sample size: 300


,id,image_path,gt
0,m01-079-03,/kaggle/input/iam-trocr/IAM/image/m01-079-03.jpg,apartment he garaged his car and then stood
1,m04-138-08,/kaggle/input/iam-trocr/IAM/image/m04-138-08.jpg,"in a moment , followed by a great"
2,n04-114-00,/kaggle/input/iam-trocr/IAM/image/n04-114-00.jpg,""" It's my business . I'll tell you more . You're"
3,e06-030-04,/kaggle/input/iam-trocr/IAM/image/e06-030-04.jpg,"course - one breeze will not do it ,"
4,m02-080-00,/kaggle/input/iam-trocr/IAM/image/m02-080-00.jpg,Sentence Database


## 8. Função de inferência

In [8]:
def predict_one_llava(image_path: str, max_new_tokens: int = 16):
    image = Image.open(image_path).convert("RGB")

    prompt = "USER: <image>\nTranscribe exactly the handwritten text in this image. Return only the transcription.\nASSISTANT:"

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output[0][prompt_len:]
    pred = processor.decode(generated_ids, skip_special_tokens=True).strip()

    return pred, 0.0

## 9. Executar tudo: inferência, métricas, análise de erros e exportação

Esta célula concentra as etapas finais para evitar erros de execução fora de ordem.


In [9]:

preds, lats = [], []

for _, row in tqdm(df_run.iterrows(), total=len(df_run), desc="Inferência"):
    pred, lat = predict_one_llava(
        row["image_path"],
        max_new_tokens=MAX_NEW_TOKENS
    )
    preds.append(pred)
    lats.append(lat)

df_run["pred_raw"] = preds
df_run["latency_ms"] = lats

# Métricas
cer = evaluate.load("cer")
wer = evaluate.load("wer")

def normalize_text(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

gt_raw = df_run["gt"].astype(str).tolist()
pred_raw = df_run["pred_raw"].astype(str).tolist()

gt_norm = [normalize_text(x) for x in gt_raw]
pred_norm = [normalize_text(x) for x in pred_raw]

metrics = {
    "CER_raw": cer.compute(predictions=pred_raw, references=gt_raw),
    "WER_raw": wer.compute(predictions=pred_raw, references=gt_raw),
    "ExactMatch_raw": sum(p == g for p, g in zip(pred_raw, gt_raw)) / len(gt_raw),
    "CER_norm": cer.compute(predictions=pred_norm, references=gt_norm),
    "WER_norm": wer.compute(predictions=pred_norm, references=gt_norm),
    "ExactMatch_norm": sum(p == g for p, g in zip(pred_norm, gt_norm)) / len(gt_norm),
    "Latency_ms_mean": float(df_run["latency_ms"].mean()),
    "Latency_ms_p95": float(df_run["latency_ms"].quantile(0.95)),
}


Inferência:   0%|          | 0/300 [00:00<?, ?it/s]

## 10. Análise de erros

In [10]:
df_run["cer_fast_norm"] = [
    Levenshtein.normalized_distance(g, p)
    for g, p in zip(gt_norm, pred_norm)
]

worst = df_run.sort_values("cer_fast_norm", ascending=False).head(20)[
    ["id", "image_path", "gt", "pred_raw", "cer_fast_norm", "latency_ms"]
]

best = df_run.sort_values("cer_fast_norm", ascending=True).head(20)[
    ["id", "image_path", "gt", "pred_raw", "cer_fast_norm", "latency_ms"]
]

## 11. Exportação

In [11]:
# Exportação
safe_model_name = MODEL_ID.replace("/", "_")
OUT_CSV = f"{safe_model_name}_iam_eval_results.csv"
OUT_TXT = f"{safe_model_name}_iam_eval_summary.txt"

df_run.to_csv(OUT_CSV, index=False)

with open(OUT_TXT, "w", encoding="utf-8") as f:
    f.write("=== IAM Handwriting Evaluation Summary ===\n")
    f.write(f"Experiment: {EXPERIMENT_NAME}\n")
    f.write(f"Model: {MODEL_ID}\n")
    f.write(f"Device: {device}\n")
    f.write(f"Sample size: {len(df_run)}\n\n")
    for k, v in metrics.items():
        f.write(f"{k}: {v}\n")
    f.write("\n--- Worst 20 ---\n")
    f.write(worst.to_string(index=False))
    f.write("\n\n--- Best 20 ---\n")
    f.write(best.to_string(index=False))

print("Resumo das métricas:")
display(pd.DataFrame([metrics]))
print("\nArquivos salvos:", OUT_CSV, OUT_TXT)
display(worst.head(10))

Resumo das métricas:


,CER_raw,WER_raw,ExactMatch_raw,CER_norm,WER_norm,ExactMatch_norm,Latency_ms_mean,Latency_ms_p95
0,0.993966,1.060837,0.0,0.966734,1.047909,0.0,0.0,0.0



Arquivos salvos: llava-hf_llava-1.5-7b-hf_iam_eval_results.csv llava-hf_llava-1.5-7b-hf_iam_eval_summary.txt


,id,image_path,gt,pred_raw,cer_fast_norm,latency_ms
249,g03-000-00,/kaggle/input/iam-trocr/IAM/image/g03-000-00.jpg,Sentence Database,J,1.0,0.0
76,g01-004-03,/kaggle/input/iam-trocr/IAM/image/g01-004-03.jpg,"leadership . But , at last , Gaunt sailed . Opposing",拼寫,1.0,0.0
188,d06-046-01,/kaggle/input/iam-trocr/IAM/image/d06-046-01.jpg,to different situations . There are some which,100,1.0,0.0
193,f04-035-01,/kaggle/input/iam-trocr/IAM/image/f04-035-01.jpg,trails . It was learned that Elizabeth Camp had been lending,1922,1.0,0.0
113,m04-113-00,/kaggle/input/iam-trocr/IAM/image/m04-113-00.jpg,"Dai , meanwhile , was pedalling",1971,1.0,0.0
121,f07-019a-02,/kaggle/input/iam-trocr/IAM/image/f07-019a-02.jpg,of the 18th century - that amazing epoch of,〇〇,1.0,0.0
219,p02-109-01,/kaggle/input/iam-trocr/IAM/image/p02-109-01.jpg,-----------------------------------------------------,The sky is white.,1.0,0.0
204,m03-110-03,/kaggle/input/iam-trocr/IAM/image/m03-110-03.jpg,"to entertain for an instant the idea , the",Mm,1.0,0.0
74,g03-004-05,/kaggle/input/iam-trocr/IAM/image/g03-004-05.jpg,were sold at 32 pieces for one shilling .,חיים,1.0,0.0
4,m02-080-00,/kaggle/input/iam-trocr/IAM/image/m02-080-00.jpg,Sentence Database,י,1.0,0.0


## 12. Baixar os arquivos no Colab

In [12]:

from google.colab import files

files.download(OUT_CSV)
files.download(OUT_TXT)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>